### Setup the entire notebook's installation and logger

In [0]:
%pip install osmnx contextily folium ipyleaflet cbsodata ydata-sdk ortools
%pip install --upgrade "typing_extensions"
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
# VUL HIER DE GEMEENTE IN WAARVOOR JE WILT DRAAIEN, let op grote gemeenten duren lang
gemeente_to_run_for = "Ede"

In [0]:
import logging
from logging.handlers import RotatingFileHandler
from pathlib import Path
from typing import Optional, Union

def setup_logging(
    level: int = logging.INFO,
    log_file: Optional[Union[str, Path]] = None,
    max_bytes: int = 10_000_000,  # 10 MB per file
    backup_count: int = 5,         # keep last 5 rotated files
) -> None:
    """
    Configure application-wide logging for use inside the notebook.
    Logs to console by default, and optionally to a rotating file.

    Parameters
    ----------
    level : int
        Logging level, e.g. logging.INFO or logging.DEBUG.
    log_file : str | Path | None
        Path to a log file. If provided, logs will be written to this file
        in addition to the console. Parent directories are created if needed.
    max_bytes : int
        Maximum size per log file before rotation (bytes).
    backup_count : int
        Number of rotated log files to keep.
    """
    root_logger = logging.getLogger()
    root_logger.setLevel(level)

    # Common formatter
    formatter = logging.Formatter(
        "%(asctime)s %(levelname)s %(name)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    # Prevent duplicate handlers on repeated cell execution
    if not any(isinstance(h, logging.StreamHandler) and not isinstance(h, logging.FileHandler)
               for h in root_logger.handlers):
        console_handler = logging.StreamHandler()
        console_handler.setFormatter(formatter)
        root_logger.addHandler(console_handler)

    if log_file is not None:
        log_path = Path(log_file)
        log_path.parent.mkdir(parents=True, exist_ok=True)

        # Only add file handler once per exact path
        existing_file_handlers = [
            h for h in root_logger.handlers
            if isinstance(h, logging.FileHandler) and getattr(h, 'baseFilename', None) == str(log_path.resolve())
        ]
        if not existing_file_handlers:
            file_handler = RotatingFileHandler(
                filename=str(log_path),
                maxBytes=max_bytes,
                backupCount=backup_count,
                encoding="utf-8"
            )
            file_handler.setFormatter(formatter)
            root_logger.addHandler(file_handler)

setup_logging(
    level=logging.INFO,
    log_file=r"/Workspace/Shared/nooddrinkwater_locaties_distributie/nooddrinkwater_locaties_distributie/logs/osm_parkeerplaatsen_EDA.log"
)
logger = logging.getLogger(__name__)

# preventing py4j info logging by setting level higher
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)


In [0]:
from IPython.display import display, HTML
from eda_support_files.CONSTANTS import DATA_EXTERNAL_FOLDER_LOCATION, DATA_INTERIM_FOLDER_LOCATION, DATA_PROCESSED_FOLDER_LOCATION, DATA_RAW_FOLDER_LOCATION

# Add all necessary variables
ASSIGNED_RESIDENT_FOLDER_LOCATION = f"{DATA_INTERIM_FOLDER_LOCATION}/gemeente_residents_assigned"
STATLINE_DIR = Path(f"{DATA_EXTERNAL_FOLDER_LOCATION}/statline_85618NED")
RUN_VISUALISATIONS = False

logging.getLogger("py4j").disabled = True

### Collecting and filtering the OpenStreetMap dataset for parking lots

In [0]:
from __future__ import annotations

import logging
from typing import Optional

import numpy as np
import geopandas as gpd

from eda_support_files.GetOSMData import GetOSMData


logger = logging.getLogger(__name__)
logger.info("Starting to load OpenStreetMap data")

# --- Load OSM data -----------------------------------------------------------

osm_data_getter = GetOSMData()
gdf_all: gpd.GeoDataFrame = osm_data_getter.run(
    folder=DATA_EXTERNAL_FOLDER_LOCATION,
    only_load=True
)

# Area of each geometry in square meters
if "area_m2" not in gdf_all.columns:
    gdf_all["area_m2"] = gdf_all.geometry.area.round(4)

# Perimeter / boundary length in meters
if "perimeter_m" not in gdf_all.columns:
    gdf_all["perimeter_m"] = gdf_all.geometry.length

# Polsby–Popper compactness measure "how square a parking lot is", to prevent long and small lots.
# (4πA) / P² where A = area, P = perimeter
if "compactness" not in gdf_all.columns:
    gdf_all["compactness"] = (
        4 * np.pi * gdf_all["area_m2"]
    ) / (gdf_all["perimeter_m"] ** 2)


#### Filtering the OSM parking lots to certain types (e.g. suface)

In [0]:
from eda_support_files.FilterOSMData import FilterOSMData

osm_data_filterer = FilterOSMData()
gdf_parking_lots = osm_data_filterer.run(gdf_all, folder=DATA_INTERIM_FOLDER_LOCATION, only_load=False)

# Fix crs for everything else after calculating area
gdf_parking_lots = gdf_parking_lots.to_crs(epsg=4326)

### Collecting and processing Gemeenten/wijken/buurten data (incl. geometry) via PDOK

In [0]:
from eda_support_files.GetGemeenteDataPDOK import GetGemeenteDataPDOK

gemeente_data_getter = GetGemeenteDataPDOK(filepath=DATA_RAW_FOLDER_LOCATION)
gdf_gemeenten_buurten = gemeente_data_getter.run(load_from_file=True)

display(gdf_gemeenten_buurten.sample(5))

In [0]:
import pandas as pd
from pathlib import Path

# --- CONFIG ---
left_df = gdf_gemeenten_buurten.copy()

# --- 1) Load Observations ---
obs = pd.read_csv(STATLINE_DIR / "Observations.csv",
                  sep=";", encoding="utf-8-sig", dtype=str)
obs = obs[["WijkenEnBuurten", "Measure", "Value"]].copy()
obs["WijkenEnBuurten"] = obs["WijkenEnBuurten"].str.strip()
obs["Measure"] = obs["Measure"].str.strip()

# --- 1.5) Value's can have a ',' which is a problem for pandas, so lets replace them with '.'
obs["Value"] = obs["Value"].str.replace(",", ".")
obs["Value"] = pd.to_numeric(obs["Value"], errors="coerce")

# Drop duplicates (region + measure)
obs = obs.drop_duplicates(subset=["WijkenEnBuurten", "Measure"])

# --- 2) Pivot to wide ---
wide = (
    obs.pivot_table(
        index="WijkenEnBuurten",
        columns="Measure",
        values="Value",
        aggfunc="first"
    )
    .reset_index()
)

# --- 3) Load MeasureCodes and MeasureGroups ---
measures = pd.read_csv(STATLINE_DIR / "MeasureCodes.csv",
                       sep=";", encoding="utf-8-sig", dtype=str)[["Identifier", "Title", "MeasureGroupId"]]
groups = pd.read_csv(STATLINE_DIR / "MeasureGroups.csv",
                     sep=";", encoding="utf-8-sig", dtype=str)[["Id", "Title", "ParentId"]]
groups = groups.rename(columns={"Id": "MeasureGroupId", "Title": "GroupTitle"})

# Build lookups
group_lookup = dict(zip(groups["MeasureGroupId"], groups["GroupTitle"]))
parent_lookup = dict(zip(groups["MeasureGroupId"], groups["ParentId"]))
parent_title_lookup = dict(zip(groups["MeasureGroupId"], groups["GroupTitle"]))

# --- 4) Build combined label: Parent - Group - Measure ---
def build_full_label(row):
    group_id = row["MeasureGroupId"]
    group_title = group_lookup.get(group_id, "")
    parent_id = parent_lookup.get(group_id)
    parent_title = parent_title_lookup.get(parent_id, "") if pd.notna(parent_id) else ""
    parts = [p for p in [parent_title, group_title, row["Title"]] if p]
    return " - ".join(parts)

label_map = {row["Identifier"]: build_full_label(row) for _, row in measures.iterrows()}

# Rename columns in wide DataFrame
wide = wide.rename(columns=label_map)


# --- 5) Merge with your left_df ---
gdf_gemeenten_buurten_dem = left_df.merge(wide, left_on="buurtcode", right_on="WijkenEnBuurten", how="left")
gdf_gemeenten_buurten_dem = gdf_gemeenten_buurten_dem.drop(columns=["WijkenEnBuurten"])
gdf_gemeenten_buurten_dem = gdf_gemeenten_buurten_dem[[
    "geometry", "buurtnaam", "buurtcode", "gemeentenaam", "aantal_inwoners", "level"
]]

display(gdf_gemeenten_buurten_dem.sample(5))
gdf_gemeenten_buurten_dem.to_csv(f"{DATA_PROCESSED_FOLDER_LOCATION}/combined_dataset.csv", index=False)

### Calculate the necessary amounts of drinkwaterpunten per Gemeente

In [0]:
from eda_support_files.CalcBenodigdWaterpunt import CalcBenodigdWaterpunt

calc_benodigd_waterpunt_procesessor = CalcBenodigdWaterpunt()
gdf_gemeenten_buurten_dem = calc_benodigd_waterpunt_procesessor.run(gdf_gemeenten_buurten_dem)

# Separate dataframes for gemeenten and buurten
gdf_gemeenten = gdf_gemeenten_buurten_dem[gdf_gemeenten_buurten_dem['level'] == 'gemeente']
gdf_buurten = gdf_gemeenten_buurten_dem[gdf_gemeenten_buurten_dem['level'] == 'buurt']

# Drop columns not used with gemeenten
gdf_gemeenten = gdf_gemeenten.drop(['buurtnaam'], axis=1)

display(gdf_buurten.tail(5))

### Select parking lots within the gemeente perimeter, sum how many in each gemeente and whether that is enough.

In [0]:
from eda_support_files.CombineGemeenteAndParkinglotData import CombineGemeenteAndParkinglotData

gemeente_parking_combiner = CombineGemeenteAndParkinglotData()
gdf_parking_lots, gdf_gemeenten = gemeente_parking_combiner.run(gdf_parking_lots, gdf_gemeenten)

display(gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == gemeente_to_run_for].sample(5))

### Let's spread the number of residents per "buurt" directly proportional over its geometry
We choose to do it on a buurt level given gemeente could have very sparcely populated areas that we dont want to spread as much residents over (e.g. Ede, having a large part "veluwe" where very litte people live)

In [0]:
from eda_support_files.GenerateGemeenteResidents import GenerateGemeenteResidents

residents_folder = f"{DATA_INTERIM_FOLDER_LOCATION}/gemeente_residents"

gemeente_residents_generator = GenerateGemeenteResidents(residents_folder)
result_text = gemeente_residents_generator.run(
    gdf_gemeenten[gdf_gemeenten['gemeentenaam'] == gemeente_to_run_for][['gemeentenaam']], gdf_buurten, overwrite=True
)

#### Load which Gemeenten are in which VeiligheidsRegios

In [0]:
import pandas as pd
gem_in_vr_pv = pd.read_csv(f"{DATA_INTERIM_FOLDER_LOCATION}/gem_in_vr_pv.csv", delimiter=';')
# Gemeenten for which we will provide the demo
gem = gem_in_vr_pv[
    (gem_in_vr_pv['Gemeentennaam'] == gemeente_to_run_for)
    ]
gem_list = gem['Gemeentennaam'].to_list()
# CONTROLEER HIER OF JE GEMEENTE DOOR IS GEKOMEN
# Het kan namelijk zijn dat de dataset waar we de gemeente uithalen niet overeenkomt met de andere dataset. Zo zijn er voorbeelden  van Laren en Hengelo
# gem_list = [i.replace('Laren (NH.)', 'Laren') for i in gem_list]
# gem_list = [i.replace('Hengelo (O.)', 'Hengelo') for i in gem_list]
gem.head(10)

### For each gemeente calculate the baseline and min cost flow resident assignment to parking lots

In [0]:
from eda_support_files.modelling.Orchestrator import ExperimentOrchestrator
import os

input_folder = f"{DATA_INTERIM_FOLDER_LOCATION}/gemeente_residents"

# Run for gemeenten in list
available_gemeente_file_paths = [f"{input_folder}/{gemeentennaam.replace(' ', '_')}.geojson" for gemeentennaam in gem_list]

SETUP_EXPERIMENTS = {
    "minimum_avg_distance_min_cost_flow": {"optimisation_class": "MinAvgDistanceSelector", "assignment_method": "min_cost_flow"},
    "minimum_max_distance_min_cost_flow": {"optimisation_class": "MinMaxDistanceSelector", "assignment_method": "min_cost_flow"},
}

# Instantiate and run orchestrator
orchestrator = ExperimentOrchestrator(
    gemeente_filepaths = available_gemeente_file_paths,
    setup_experiments = SETUP_EXPERIMENTS,
    output_folder = ASSIGNED_RESIDENT_FOLDER_LOCATION,
    gdf_parking_lots = gdf_parking_lots,
    gdf_gemeenten = gdf_gemeenten
)
orchestrator.run()


### Use the code below if you want to run for a specific gemeente and visualise the steps in between

In [0]:
import os
import geopandas as gpd
import pandas as pd

# Inputs
experiment_folder = os.path.join(DATA_INTERIM_FOLDER_LOCATION, "gemeente_residents_assigned", gemeente_to_run_for)

# Discover all .geojson files and load them
def load_experiments(folder: str, experiment_method: str = None) -> dict[str, gpd.GeoDataFrame]:
    """
    Loads all GeoJSON experiment files from the given folder.
    Returns a dict: {experiment_name: GeoDataFrame}
    """
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Experiment folder does not exist: {folder}")

    experiments = {}
    for fname in os.listdir(folder):
        # accept .geojson or .json, case-insensitive
        if fname.lower().endswith((".geojson", ".json")):
            experiment_name = os.path.splitext(fname)[0]  # strip extension
            # Skip if not experiment_method when given
            if experiment_method:
                if experiment_name != experiment_method:
                    continue
            fpath = os.path.join(folder, fname)
            try:
                gdf = gpd.read_file(fpath)
                experiments[experiment_name] = gdf
            except Exception as e:
                # Log and continue loading others
                logger.warning(f"[WARN] Failed to read '{fpath}': {e}")

    if not experiments:
        logger.info(f"[INFO] No experiment files found in: {folder}")

    return experiments

# Load all experiments
experiments = load_experiments(experiment_folder)

In [0]:
gdf_parking_lots_gem = gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == gemeente_to_run_for]
gdf_gemeenten_gem = gdf_gemeenten[gdf_gemeenten['gemeentenaam'] == gemeente_to_run_for]

display(HTML(f"<h2>{gemeente_to_run_for} has {len(experiments.get("minimum_avg_distance_min_cost_flow"))} residents</h2>"))
display(HTML(f"<h2>{gemeente_to_run_for} has {gdf_gemeenten_gem["aantal_parkeerplaatsen"].iloc[0]} eligible parking lots for nooddrinkwaterpunten and needs {gdf_gemeenten_gem["Benodigd_ceiling"].iloc[0]}</h2>"))

display(HTML("<h3>Here is a sample of the available parking lots:</h3>"))

### Summary of evaluation metrics

In [0]:
from eda_support_files.summarize_results import summarize_assignments, visualize_distances, get_np_bins
from IPython.display import display, HTML
import plotly.graph_objects as go
import pandas as pd

# --- Build global bins across ALL experiments so histograms align
all_distances = pd.concat(
    [gdf["distance_to_parking"] for gdf in experiments.values()],
    ignore_index=True
)

# If get_np_bins expects two arrays, we can pass the same twice to derive bins from the whole set.
# Otherwise, if it accepts an iterable, change accordingly.
np_bins = get_np_bins(all_distances, all_distances)

# --- Create figure and plot each experiment
fig = go.Figure()

all_max_y = []
for exp_name, gdf in sorted(experiments.items()):
    display(HTML(f"<h3>Experiment: {exp_name}</h3>"))
    # Summary (table/metrics)
    summarize_assignments(gdf, gdf["distance_to_parking"])

    # Add histogram/trace to figure
    fig, max_y = visualize_distances(
        fig,
        gdf["distance_to_parking"],
        np_bins=np_bins,
        name=exp_name
    )
    all_max_y.append(max_y)
# Add the line for loopafstand
fig.add_shape(type="line", x0=1000, x1=1000, y0=0, y1=max(all_max_y), opacity=1,
                line=dict(color="black", width=4))
# --- Final layout for the combined plot
fig.update_layout(
    title="Distribution of resident-to-parking-lot distances (all experiments)",
    xaxis_title="Distance (m)",
    showlegend=True,
    bargap=0.1
)

fig.show()


## Export to GPKG

In [0]:
from eda_support_files.create_visualisation import create_interactive_map
import matplotlib.colors as mcolors
from matplotlib import colormaps
import numpy as np
import random

experiment_method_to_use = "minimum_avg_distance_min_cost_flow"
# experiment_method_to_use = "minimum_max_distance_min_cost_flow"

residents_nh = []
parking_lots_nh = []

for gemeente in gem_list:
    experiment_folder = os.path.join(DATA_INTERIM_FOLDER_LOCATION, "gemeente_residents_assigned", gemeente)
    experiments = load_experiments(experiment_folder, experiment_method=experiment_method_to_use)

    # === Get residents data ===
    gdf_res_plot = experiments.get(experiment_method_to_use).copy()

    # === Prepare color mapping ===
    unique_lots = gdf_res_plot['assigned_parking_lot'].dropna().unique()
    random.shuffle(unique_lots)

    # Get colormap and sample colors
    cmap = colormaps.get_cmap('gist_rainbow')
    colors = [mcolors.to_hex(cmap(x)) for x in np.linspace(0, 1, len(unique_lots))]

    # Build mapping: lot ID → color
    lot_to_color = {int(lot): color for lot, color in zip(unique_lots, colors)}

    # Add color column to residents
    gdf_res_plot['color'] = gdf_res_plot['assigned_parking_lot'].map(lot_to_color)

    # === Prepare parking lots ===
    gdf_parking_lots_gem = gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == gemeente]
    gdf_parking_lots_gem_selected = gdf_parking_lots_gem[
        gdf_parking_lots_gem.index.isin(unique_lots)
    ][['geometry']].copy()
    gdf_parking_lots_gem_selected['color'] = gdf_parking_lots_gem_selected.index.map(lot_to_color)

    ## Build GeoJSON features for parking lots
    parking_features = []
    for idx, row in gdf_parking_lots_gem_selected.iterrows():
        props = {"color": row["color"], "type": "parking_lot"}
        parking_features.append({
            "type": "Feature",
            "geometry": row.geometry.__geo_interface__,
            "properties": props
        })

    # === Config ===
    out_dir = f"{DATA_PROCESSED_FOLDER_LOCATION}/qgis_output/{experiment_method_to_use}/"
    gpkg_path = os.path.join(out_dir, "residents_parking_interview.gpkg")

    os.makedirs(out_dir, exist_ok=True)

    # === Copies of original data ===
    residents_copy = gdf_res_plot.copy()
    parking_copy = gdf_parking_lots_gem_selected.copy()

    # Ensure WGS84 CRS
    def to_wgs84(gdf):
        return (gdf.set_crs(4326) if gdf.crs is None else gdf.to_crs(4326))

    residents_copy = to_wgs84(residents_copy)
    parking_copy = to_wgs84(parking_copy)
    parking_copy = parking_copy.reset_index().rename(columns={"index": "parking_lot_id"})

    # Prepare columns for export
    residents_copy = residents_copy[["assigned_parking_lot", "color", "geometry"]]
    parking_copy = parking_copy[["parking_lot_id", "color", "geometry"]]
    # combine
    residents_nh.append(residents_copy)
    parking_lots_nh.append(parking_copy)

# combine all in list
residents_nh_comb = pd.concat(residents_nh).reset_index(drop=True)
parking_lots_nh_comb = pd.concat(parking_lots_nh).reset_index(drop=True)

# === Export GeoPackage ===
residents_nh_comb.to_file(gpkg_path, layer="residents", driver="GPKG")
parking_lots_nh_comb.to_file(gpkg_path, layer="parking_lots", driver="GPKG")

logger.info("✅ Export complete:")
logger.info(f"GeoPackage: {gpkg_path} (layers: residents, parking_lots)")
